# 12 - Word Embeddings with TensorFlow/Keras

Goal: Learn word embeddings using TensorFlow and Keras

Run with: conda activate tfenv

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0

In [2]:
vocab = ["el", "la", "gato", "perro", "come", "duerme", "en", "casa", "corre", "nada"]
vocab_size = len(vocab)
embed_dim = 3

word_to_idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocab size: {vocab_size}")
print(f"Words: {vocab}")
print(f"Word to index: {word_to_idx}")

Vocab size: 10
Words: ['el', 'la', 'gato', 'perro', 'come', 'duerme', 'en', 'casa', 'corre', 'nada']
Word to index: {'el': 0, 'la': 1, 'gato': 2, 'perro': 3, 'come': 4, 'duerme': 5, 'en': 6, 'casa': 7, 'corre': 8, 'nada': 9}

In [3]:
print("=== Method 1: tf.keras.layers.Embedding ===")
embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)

gato_idx = word_to_idx["gato"]
gato_embedding = embedding_layer(tf.constant(gato_idx))
print(f"Embedding for 'gato': {gato_embedding.numpy()}")

Embedding for 'gato': [ 0.02623371 -0.01389936 -0.03235319]

In [4]:
print("=== Method 2: Dense layer (manual) ===")
dense_layer = layers.Dense(embed_dim, use_bias=False)

gato_onehot = tf.one_hot(gato_idx, vocab_size)
dense_embedding = dense_layer(gato_onehot)
print(f"Dense embedding for 'gato': {dense_embedding.numpy()}")

print("\nBoth methods learn a weight matrix of shape:", embedding_layer.weights[0].shape)

Dense embedding for 'gato': [-0.04683795 -0.04133694 -0.02325547]

Both methods learn a weight matrix of shape: (10, 3)

In [5]:
import plotly.express as px
import pandas as pd

all_embeddings = embedding_layer(np.arange(vocab_size)).numpy()

df = pd.DataFrame(all_embeddings, columns=['x', 'y', 'z'])
df['word'] = vocab

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (Random - TensorFlow)')
fig.update_traces(marker=dict(size=8), textposition='top center')
fig.show()

plt.savefig('plots/12_embeddings_random.png', dpi=100, bbox_inches='tight')
print("Saved: plots/12_embeddings_random.png")

Saved: plots/12_embeddings_random.png

In [6]:
print("=== Training embeddings ===")

pairs = [
    (0, 2),
    (0, 3),
    (2, 3),
    (2, 8),
    (3, 8),
    (3, 9),
    (4, 5),
    (6, 7),
]

optimizer = keras.optimizers.Adam(learning_rate=0.1)
loss_fn = keras.losses.MeanSquaredError()

for epoch in range(500):
    total_loss = 0
    for word_idx, context_idx in pairs:
        word = tf.constant([word_idx])
        context = tf.constant([context_idx])

        with tf.GradientTape() as tape:
            word_emb = embedding_layer(word)
            context_emb = embedding_layer(context)
            loss = loss_fn(word_emb, context_emb * 0.5)

        gradients = tape.gradient(loss, embedding_layer.trainable_weights)
        optimizer.apply_gradients(zip(gradients, embedding_layer.trainable_weights))
        total_loss += loss

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.numpy():.4f}")

Epoch 0, Loss: 5.3134
Epoch 100, Loss: 0.0043
Epoch 200, Loss: 0.0004
Epoch 300, Loss: 0.0000
Epoch 400, Loss: 0.0000

In [7]:
trained_embeddings = embedding_layer(np.arange(vocab_size)).numpy()

df_trained = pd.DataFrame(trained_embeddings, columns=['x', 'y', 'z'])
df_trained['word'] = vocab

fig = px.scatter_3d(df_trained, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (Trained - TensorFlow)')
fig.update_traces(marker=dict(size=8, color='red'), textposition='top center')
fig.show()

plt.savefig('plots/12_embeddings_trained.png', dpi=100, bbox_inches='tight')
print("Saved: plots/12_embeddings_trained.png")

Saved: plots/12_embeddings_trained.png